Supplementary tables. Run after 1_code.ipynb (it reads core_outputs/ from that notebook).

1) Refit the development models once more, in the same loops as 1_code.ipynb, and also:
- the same forest without Platt scaling (Table S5)
- external probabilities from each fold model, averaged over the five (Table S6)
- feature counts after each step (Table S2)
- 10, 30 and 40 selected features instead of 20 (Table S10)
- a calibrated gradient-boosting model on the final 20 features (Table S9)

2) Supplemental tables, numbered in the order the manuscript first refers to them:
- S1 cohort characteristics (from the labeling notebook, 07_create_2021_glioma_labels: table1_baseline.csv, not computed here)
- S2 feature counts
- S3 model settings
- S4 base-classifier performance (from performance.csv, 1_code.ipynb)
- S5 with and without Platt scaling
- S6 rules set on development data only (transported)
- S7 Table 1 with split-to-split ranges
- S8 singleton yield and accuracy by subtype
- S9 gradient-boosting base learner
- S10 number of selected features
- S11 the 20 selected features

Supplemental figures come from the figure notebooks: Figure S1 (3_figure3.ipynb), Figure S2 (3_figure4.ipynb).
Every table is written to core_outputs/ as a csv.

Setup

In [1]:
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from itertools import product
from sklearn.model_selection import StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
from mrmr import mrmr_classif
from crepes import ConformalClassifier

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 30)

OUT = 'core_outputs'
table = pd.read_csv('modeling_table.csv')
feature_cols = [c for c in table.columns if c not in ['Scanner', '2021_glioma_class', 'age', 'sex', 'case_id', 'site']]
df_dev = table[table['site'].isin(['TCGA', 'UCSF'])]
y_dev = df_dev['2021_glioma_class'].values
y_egd = table[table['site'] == 'EGD']['2021_glioma_class'].values
y_utsw = table[table['site'] == 'UTSW']['2021_glioma_class'].values
classes = np.unique(y_dev)
X_dev = df_dev[feature_cols].replace([np.inf, -np.inf], np.nan)
X_egd = table[table['site'] == 'EGD'][feature_cols].replace([np.inf, -np.inf], np.nan)
X_utsw = table[table['site'] == 'UTSW'][feature_cols].replace([np.inf, -np.inf], np.nan)

# same random forest as 1_code.ipynb
rf = RandomForestClassifier(n_estimators=500, class_weight='balanced_subsample', max_features='sqrt', random_state=42, n_jobs=-1)
rf_platt = CalibratedClassifierCV(estimator=rf, method='sigmoid', cv=5)

# gradient boosting (fixed settings, not tuned), Platt scaling with 5 inner folds, balanced sample weights
gb = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_depth=3, l2_regularization=1.0, random_state=42)
gb_platt = CalibratedClassifierCV(estimator=gb, method='sigmoid', cv=5)

k_values = [10, 20, 30, 40]

1) Development 5-fold cross-validation, same folds and steps as 1_code.ipynb. In each fold:
- the calibrated forest with 20 features (must equal oof_probas.npy from 1_code.ipynb)
- the same forest without Platt scaling
- the calibrated forest with 10, 30 and 40 features
- EGD and UTSW probabilities from the fold model, averaged over the five folds

In [2]:
oof_uncal = np.zeros((len(y_dev), 3))
oof_k = {K: np.zeros((len(y_dev), 3)) for K in k_values}
egd_fold = np.zeros((len(y_egd), 3))
utsw_fold = np.zeros((len(y_utsw), 3))
count_rows = []
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr, va) in enumerate(skf.split(X_dev, y_dev)):
    # imputation
    imp = SimpleImputer(strategy='median')
    X_tr = pd.DataFrame(imp.fit_transform(X_dev.iloc[tr]), columns=feature_cols)
    X_va = pd.DataFrame(imp.transform(X_dev.iloc[va]), columns=feature_cols)
    X_e = pd.DataFrame(imp.transform(X_egd), columns=feature_cols)
    X_u = pd.DataFrame(imp.transform(X_utsw), columns=feature_cols)

    # variance filter, raw scale, then z-score
    keep = X_tr.columns[X_tr.var() > 1e-5]
    scaler = StandardScaler()
    X_tr = pd.DataFrame(scaler.fit_transform(X_tr[keep]), columns=keep)
    X_va = pd.DataFrame(scaler.transform(X_va[keep]), columns=keep)
    X_e = pd.DataFrame(scaler.transform(X_e[keep]), columns=keep)
    X_u = pd.DataFrame(scaler.transform(X_u[keep]), columns=keep)

    # correlation filter
    upper = X_tr.corr().abs().where(np.triu(np.ones((X_tr.shape[1], X_tr.shape[1])), k=1).astype(bool))
    X_tr = X_tr.drop(columns=[c for c in upper.columns if any(upper[c] > 0.85)])
    count_rows.append({'Model': 'CV fold ' + str(fold + 1), 'Training patients': len(tr), 'Initial': len(feature_cols),
                       'After variance filter': len(keep), 'After correlation filter': X_tr.shape[1], 'After mRMR': 20})

    # calibrated forest for each number of features (K = 20 is the main model)
    for K in k_values:
        features = mrmr_classif(X=X_tr, y=pd.Series(y_dev[tr]), K=K, show_progress=False)
        model = clone(rf_platt)
        model.fit(X_tr[features].values, y_dev[tr])
        oof_k[K][va] = model.predict_proba(X_va[features].values)
        if K == 20:
            egd_fold = egd_fold + model.predict_proba(X_e[features].values) / 5      # average of the five fold models
            utsw_fold = utsw_fold + model.predict_proba(X_u[features].values) / 5
            model_uncal = clone(rf)                                                  # same forest, no Platt scaling
            model_uncal.fit(X_tr[features].values, y_dev[tr])
            oof_uncal[va] = model_uncal.predict_proba(X_va[features].values)

    print('fold', fold + 1, 'done')

# check: K = 20 must reproduce 1_code.ipynb exactly
print('calibrated OOF identical to 1_code.ipynb:', np.array_equal(oof_k[20], np.load(os.path.join(OUT, 'oof_probas.npy'))))

fold 1 done
fold 2 done
fold 3 done
fold 4 done
fold 5 done
calibrated OOF identical to 1_code.ipynb: True


2) Final model on all 611 development patients, same steps as 1_code.ipynb:
- the same forest without Platt scaling
- the calibrated gradient-boosting model on the same 20 features

In [3]:
imp = SimpleImputer(strategy='median')
X_tr = pd.DataFrame(imp.fit_transform(X_dev), columns=feature_cols)
X_e = pd.DataFrame(imp.transform(X_egd), columns=feature_cols)
X_u = pd.DataFrame(imp.transform(X_utsw), columns=feature_cols)

keep = X_tr.columns[X_tr.var() > 1e-5]
scaler = StandardScaler()
X_tr = pd.DataFrame(scaler.fit_transform(X_tr[keep]), columns=keep)
X_e = pd.DataFrame(scaler.transform(X_e[keep]), columns=keep)
X_u = pd.DataFrame(scaler.transform(X_u[keep]), columns=keep)

upper = X_tr.corr().abs().where(np.triu(np.ones((X_tr.shape[1], X_tr.shape[1])), k=1).astype(bool))
X_tr = X_tr.drop(columns=[c for c in upper.columns if any(upper[c] > 0.85)])
count_rows.append({'Model': 'Final model', 'Training patients': len(y_dev), 'Initial': len(feature_cols),
                   'After variance filter': len(keep), 'After correlation filter': X_tr.shape[1], 'After mRMR': 20})

final_features = mrmr_classif(X=X_tr, y=pd.Series(y_dev), K=20, show_progress=False)

final_uncal = clone(rf)
final_uncal.fit(X_tr[final_features], y_dev)
egd_uncal = final_uncal.predict_proba(X_e[final_features])
utsw_uncal = final_uncal.predict_proba(X_u[final_features])

final_gb = clone(gb_platt)
final_gb.fit(X_tr[final_features].values, y_dev, sample_weight=compute_sample_weight('balanced', y_dev))
egd_gb = final_gb.predict_proba(X_e[final_features].values)
utsw_gb = final_gb.predict_proba(X_u[final_features].values)

np.save(os.path.join(OUT, 'oof_probas_uncal.npy'), oof_uncal)
np.save(os.path.join(OUT, 'egd_probas_uncal.npy'), egd_uncal)
np.save(os.path.join(OUT, 'utsw_probas_uncal.npy'), utsw_uncal)
np.save(os.path.join(OUT, 'egd_probas_foldavg.npy'), egd_fold)
np.save(os.path.join(OUT, 'utsw_probas_foldavg.npy'), utsw_fold)
np.save(os.path.join(OUT, 'egd_probas_gb.npy'), egd_gb)
np.save(os.path.join(OUT, 'utsw_probas_gb.npy'), utsw_gb)

# check: same 20 features as 1_code.ipynb (its final model must give the same EGD probabilities)
print('final features (first 3):', final_features[:3])

final features (first 3): ['T1CE_original_firstorder_90Percentile', 'T1_wavelet-LLL_glszm_ZoneEntropy', 'T2_log-sigma-2-0-mm-3D_glszm_GrayLevelNonUniformityNormalized']


Table S2: number of features after each step, for each fold and the final model.

In [4]:
tableS2 = pd.DataFrame(count_rows)
tableS2.to_csv(os.path.join(OUT, 'tableS2_feature_counts.csv'), index=False)
print(tableS2.to_string(index=False))

      Model  Training patients  Initial  After variance filter  After correlation filter  After mRMR
  CV fold 1                488     4224                   3841                       805          20
  CV fold 2                489     4224                   3842                       806          20
  CV fold 3                489     4224                   3843                       807          20
  CV fold 4                489     4224                   3842                       804          20
  CV fold 5                489     4224                   3844                       798          20
Final model                611     4224                   3843                       814          20


Table S3: model settings. Every parameter of the forest and of the gradient-boosting model, and the calibration settings.

In [5]:
settings_rf = pd.Series(rf.get_params()).astype(str)
settings_gb = pd.Series(gb.get_params()).astype(str)
tableS3 = pd.concat([pd.DataFrame({'Model': 'Random forest', 'Parameter': settings_rf.index, 'Value': settings_rf.values}),
                     pd.DataFrame({'Model': 'Gradient boosting', 'Parameter': settings_gb.index, 'Value': settings_gb.values})])
tableS3 = pd.concat([tableS3, pd.DataFrame([{'Model': 'Both', 'Parameter': 'calibration method', 'Value': 'sigmoid (Platt)'},
                                            {'Model': 'Both', 'Parameter': 'calibration inner folds', 'Value': '5'},
                                            {'Model': 'Gradient boosting', 'Parameter': 'sample weights', 'Value': 'balanced'}])])
tableS3.to_csv(os.path.join(OUT, 'tableS3_settings.csv'), index=False)
print(tableS3.to_string(index=False))

            Model                Parameter              Value
    Random forest                bootstrap               True
    Random forest                ccp_alpha                0.0
    Random forest             class_weight balanced_subsample
    Random forest                criterion               gini
    Random forest                max_depth               None
    Random forest             max_features               sqrt
    Random forest           max_leaf_nodes               None
    Random forest              max_samples               None
    Random forest    min_impurity_decrease                0.0
    Random forest         min_samples_leaf                  1
    Random forest        min_samples_split                  2
    Random forest min_weight_fraction_leaf                0.0
    Random forest            monotonic_cst               None
    Random forest             n_estimators                500
    Random forest                   n_jobs                 -1
    Rand

Table S4: base-classifier performance, per subtype (one against the rest). Point estimate on the full cohort, 95% CI from 1000 bootstrap samples (from performance.csv, written by 1_code.ipynb). Sensitivity to NPV use the argmax label.

In [6]:
perf = pd.read_csv(os.path.join(OUT, 'performance.csv'))
s4_rows = []
for i in range(len(perf)):
    r = perf.iloc[i]
    row = {'Cohort': r['cohort'], 'Subtype': r['class']}
    for name in ['AUC', 'AUPRC', 'Sens', 'Spec', 'PPV', 'NPV']:
        row[name] = '{:.2f} ({:.2f}-{:.2f})'.format(r[name], r[name + '_lo'], r[name + '_hi'])
    s4_rows.append(row)
tableS4 = pd.DataFrame(s4_rows)
tableS4.to_csv(os.path.join(OUT, 'tableS4_performance.csv'), index=False)
print(tableS4.to_string(index=False))

 Cohort           Subtype              AUC            AUPRC             Sens             Spec              PPV              NPV
Dev OOF       astrocytoma 0.95 (0.93-0.97) 0.88 (0.83-0.92) 0.79 (0.72-0.86) 0.95 (0.93-0.97) 0.81 (0.74-0.87) 0.94 (0.92-0.96)
Dev OOF      glioblastoma 0.96 (0.94-0.98) 0.98 (0.97-0.99) 0.98 (0.97-0.99) 0.79 (0.72-0.85) 0.93 (0.90-0.95) 0.93 (0.89-0.97)
Dev OOF oligodendroglioma 0.88 (0.81-0.93) 0.34 (0.20-0.53) 0.12 (0.00-0.25) 1.00 (0.99-1.00) 0.60 (0.00-1.00) 0.96 (0.95-0.98)
    EGD       astrocytoma 0.92 (0.89-0.95) 0.67 (0.56-0.79) 0.79 (0.69-0.87) 0.86 (0.82-0.90) 0.58 (0.47-0.67) 0.95 (0.92-0.97)
    EGD      glioblastoma 0.96 (0.95-0.98) 0.98 (0.97-0.99) 0.96 (0.94-0.98) 0.71 (0.63-0.78) 0.85 (0.81-0.89) 0.91 (0.86-0.96)
    EGD oligodendroglioma 0.87 (0.83-0.91) 0.52 (0.42-0.65) 0.10 (0.03-0.17) 0.99 (0.98-1.00) 0.70 (0.38-1.00) 0.83 (0.79-0.87)
   UTSW       astrocytoma 0.90 (0.87-0.94) 0.60 (0.48-0.72) 0.70 (0.60-0.81) 0.92 (0.89-0.94) 0.59 (0.48

Table S5: the random forest with and without Platt scaling (reviewer 2.10).
Development values are out-of-fold. External values are the final model on the full cohort.
Sensitivity uses the argmax label. Calibration error (ECE): top label, 15 equal-width bins, weighted by bin size.

In [7]:
edges = np.linspace(0, 1, 16)
s3_rows = []
for cohort, y, name in [('Development (out-of-fold)', y_dev, 'oof'), ('EGD', y_egd, 'egd'), ('UTSW-Glioma', y_utsw, 'utsw')]:
    for version, suffix in [('With Platt scaling', ''), ('Without Platt scaling', '_uncal')]:
        p = np.load(os.path.join(OUT, name + '_probas' + suffix + '.npy'))
        pred = classes[np.argmax(p, axis=1)]
        is_o = y == 'oligodendroglioma'

        # top-label calibration error: gap between confidence and accuracy in each bin
        conf = p.max(axis=1)
        hit = (pred == y).astype(float)
        bin_of = np.clip(np.digitize(conf, edges[1:-1], right=True), 0, 14)     # bin 0 to 14 for each patient
        ece = 0.0
        for b in range(15):
            in_bin = bin_of == b
            if in_bin.any():
                ece = ece + in_bin.mean() * abs(hit[in_bin].mean() - conf[in_bin].mean())

        s3_rows.append({'Cohort': cohort, 'Probabilities': version,
                        'Macro AUC': round(roc_auc_score(y, p, multi_class='ovr', labels=classes), 3),
                        'AUC, astrocytoma': round(roc_auc_score(y == classes[0], p[:, 0]), 3),
                        'AUC, glioblastoma': round(roc_auc_score(y == classes[1], p[:, 1]), 3),
                        'AUC, oligodendroglioma': round(roc_auc_score(is_o, p[:, 2]), 3),
                        'Sensitivity, astrocytoma': round(np.mean(pred[y == classes[0]] == classes[0]), 3),
                        'Sensitivity, glioblastoma': round(np.mean(pred[y == classes[1]] == classes[1]), 3),
                        'Sensitivity, oligodendroglioma': round(np.mean(pred[is_o] == classes[2]), 3),
                        'Oligodendrogliomas called correctly': str(np.sum(pred[is_o] == classes[2])) + '/' + str(np.sum(is_o)),
                        'Patients called oligodendroglioma': int(np.sum(pred == classes[2])),
                        'Calibration error (ECE)': round(ece, 3)})

tableS5 = pd.DataFrame(s3_rows)
tableS5.to_csv(os.path.join(OUT, 'tableS5_platt.csv'), index=False)
print(tableS5.to_string(index=False))

                   Cohort         Probabilities  Macro AUC  AUC, astrocytoma  AUC, glioblastoma  AUC, oligodendroglioma  Sensitivity, astrocytoma  Sensitivity, glioblastoma  Sensitivity, oligodendroglioma Oligodendrogliomas called correctly  Patients called oligodendroglioma  Calibration error (ECE)
Development (out-of-fold)    With Platt scaling      0.928             0.949              0.957                   0.878                     0.789                      0.980                           0.115                                3/26                                  5                    0.050
Development (out-of-fold) Without Platt scaling      0.927             0.948              0.958                   0.876                     0.789                      0.980                           0.077                                2/26                                  3                    0.049
                      EGD    With Platt scaling      0.919             0.920              0.963  

Table S6: rules set on development data only, applied unchanged to the external cohorts (reviewer 2.9).
Nothing is fitted on external patients. Scored on the same 200 test halves as Table 1, at the 90% target.
- Argmax: the model's top subtype
- Class-weighted argmax: weights chosen on development out-of-fold probabilities (same 729-weight grid as 1_code.ipynb)
- Probability sets: subtypes added until the model probabilities reach 90% (no recalibration)
- Class-conditional conformal, transported: per-subtype cutoffs from development out-of-fold scores, applied to the final-model probabilities, and (check) to the average of the five fold models, which produced those out-of-fold scores

Also: the 90% cutoff per subtype from development scores and from each full external cohort (score shift).
Cutoff rule, same as crepes: the largest calibration score with (number of scores at or above it, plus 1) / (n + 1) >= 1 - target. A subtype is listed when its score (1 - probability) is at or below its cutoff.

In [8]:
level = 0.90
eye3 = np.eye(3, dtype=int)
oof = np.load(os.path.join(OUT, 'oof_probas.npy'))
dev_col = np.searchsorted(classes, y_dev)

# development class weights, fixed before looking at any external patient
weight_grid = [np.array(w, dtype=float) for w in product([0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0], repeat=3)]
bal = [balanced_accuracy_score(dev_col, np.argmax(oof * w, axis=1)) for w in weight_grid]
dev_w = weight_grid[int(np.argmax(bal))]
print('development class weights (A, G, O):', dev_w)

# development cutoff per subtype
dev_scores = 1 - oof[np.arange(len(y_dev)), dev_col]
dev_cut = []
for k in range(3):
    sc = dev_scores[dev_col == k]
    dev_cut.append(max(s for s in sc if (np.sum(sc >= s) + 1) / (len(sc) + 1) >= 1 - level))
dev_cut = np.array(dev_cut)

s5_rows = []
cut_rows = [{'Score source': 'Development (out-of-fold)', 'Astrocytoma': dev_cut[0], 'Glioblastoma': dev_cut[1], 'Oligodendroglioma': dev_cut[2]}]
for cohort, y in [('EGD', y_egd), ('UTSW', y_utsw)]:
    col = np.searchsorted(classes, y)
    p = np.load(os.path.join(OUT, cohort.lower() + '_probas.npy'))
    p_fold = np.load(os.path.join(OUT, cohort.lower() + '_probas_foldavg.npy'))
    is_test = np.load(os.path.join(OUT, 'is_test_' + cohort.lower() + '.npy'))

    # sets for every patient of the cohort (no split is needed to build them)
    order = np.argsort(-p, axis=1)
    sorted_p = np.take_along_axis(p, order, axis=1)
    set_prob = np.zeros(p.shape, dtype=int)
    np.put_along_axis(set_prob, order, ((np.cumsum(sorted_p, axis=1) - sorted_p) < level).astype(int), axis=1)
    rule_sets = [('Argmax', eye3[np.argmax(p, axis=1)]),
                 ('Class-weighted argmax, development weights', eye3[np.argmax(p * dev_w, axis=1)]),
                 ('Probability sets, model probabilities', set_prob),
                 ('Class-conditional conformal, transported cutoffs (final model)', ((1 - p) <= dev_cut).astype(int)),
                 ('Class-conditional conformal, transported cutoffs (fold models)', ((1 - p_fold) <= dev_cut).astype(int))]

    # score each rule on the same 200 test halves as Table 1
    for seed in range(200):
        test = np.where(is_test[seed])[0]
        yt = col[test]
        for rule, s_all in rule_sets:
            s = s_all[test]
            covered = s[np.arange(len(test)), yt] == 1
            size = s.sum(axis=1)
            single = size == 1
            s5_rows.append({'cohort': cohort, 'seed': seed, 'rule': rule,
                            'cov_A': covered[yt == 0].mean(), 'cov_G': covered[yt == 1].mean(), 'cov_O': covered[yt == 2].mean(),
                            'size': size.mean(), 'single_pct': single.mean() * 100,
                            'acc_single_pct': covered[single].mean() * 100 if single.any() else np.nan})

    # the cutoff this full external cohort would give, to show the shift
    ext_scores = 1 - p[np.arange(len(y)), col]
    ext_cut = []
    for k in range(3):
        sc = ext_scores[col == k]
        ext_cut.append(max(s for s in sc if (np.sum(sc >= s) + 1) / (len(sc) + 1) >= 1 - level))
    cut_rows.append({'Score source': cohort + ' (full cohort)', 'Astrocytoma': ext_cut[0], 'Glioblastoma': ext_cut[1], 'Oligodendroglioma': ext_cut[2]})

s5 = pd.DataFrame(s5_rows)
means = s5.groupby(['cohort', 'rule'], sort=False).mean(numeric_only=True).drop(columns='seed').reset_index()
tableS6 = pd.DataFrame({'Cohort': means['cohort'], 'Rule': means['rule'],
                        'Coverage, astrocytoma': means['cov_A'].round(2), 'Coverage, glioblastoma': means['cov_G'].round(2),
                        'Coverage, oligodendroglioma': means['cov_O'].round(2), 'Mean set size': means['size'].round(2),
                        'Singleton yield (%)': means['single_pct'].round(1), 'Singleton accuracy (%)': means['acc_single_pct'].round(1)})
tableS6.to_csv(os.path.join(OUT, 'tableS6_transported.csv'), index=False)
print(tableS6.to_string(index=False))

cutoffs = pd.DataFrame(cut_rows).round(3)
cutoffs.to_csv(os.path.join(OUT, 'tableS6_cutoffs.csv'), index=False)
print('\n90% conformal cutoff (score = 1 - probability of the true subtype):')
print(cutoffs.to_string(index=False))

development class weights (A, G, O): [1.5 0.5 6. ]
Cohort                                                           Rule  Coverage, astrocytoma  Coverage, glioblastoma  Coverage, oligodendroglioma  Mean set size  Singleton yield (%)  Singleton accuracy (%)
   EGD                                                         Argmax                   0.80                    0.96                         0.09           1.00                100.0                    77.3
   EGD                     Class-weighted argmax, development weights                   0.76                    0.87                         0.53           1.00                100.0                    78.8
   EGD                          Probability sets, model probabilities                   1.00                    1.00                         0.61           1.66                 47.1                    97.8
   EGD Class-conditional conformal, transported cutoffs (final model)                   0.99                    0.84         

Table S7: Table 1 with split-to-split ranges, and how often oligodendroglioma was listed for patients without it.
Each cell: mean over the 200 test halves [2.5th to 97.5th percentile across splits]. Reads decision_rules_200_splits.csv.

In [9]:
rules = pd.read_csv(os.path.join(OUT, 'decision_rules_200_splits.csv'))

# rule name in the csv, rule name in the manuscript
rule_order = [('Recalibrated argmax', 'Recalibrated argmax'),
              ('Locally weighted argmax', 'Class-weighted argmax'),
              ('Recalibrated probability sets', 'Probability sets'),
              ('Probability abstention', 'Probability-based abstention'),
              ('Single-threshold conformal', 'Standard conformal'),
              ('Mondrian conformal', 'Class-conditional conformal')]
s4_columns = [('cov_A', 'Coverage, astrocytoma', 2), ('cov_G', 'Coverage, glioblastoma', 2), ('cov_O', 'Coverage, oligodendroglioma', 2),
              ('size', 'Mean set size', 2), ('O_listed_for_nonO', 'Oligodendroglioma listed, non-oligodendroglioma patients', 2),
              ('single_pct', 'Singleton yield (%)', 1), ('acc_single_pct', 'Singleton accuracy (%)', 1)]

s4_rows = []
for cohort in ['EGD', 'UTSW']:
    for rule, name in rule_order:
        d = rules[(rules['cohort'] == cohort) & (rules['rule'] == rule)]
        row = {'Cohort': cohort, 'Rule': name}
        for col, label, digits in s4_columns:
            fmt = '{:.' + str(digits) + 'f}'
            row[label] = fmt.format(d[col].mean()) + ' [' + fmt.format(d[col].quantile(0.025)) + ' to ' + fmt.format(d[col].quantile(0.975)) + ']'
        s4_rows.append(row)

tableS7 = pd.DataFrame(s4_rows)
tableS7.to_csv(os.path.join(OUT, 'tableS7_ranges.csv'), index=False)
print(tableS7.to_string(index=False))

Cohort                         Rule Coverage, astrocytoma Coverage, glioblastoma Coverage, oligodendroglioma       Mean set size Oligodendroglioma listed, non-oligodendroglioma patients    Singleton yield (%) Singleton accuracy (%)
   EGD          Recalibrated argmax   0.67 [0.54 to 0.79]    0.94 [0.91 to 0.98]         0.32 [0.19 to 0.46] 1.00 [1.00 to 1.00]                                      0.07 [0.04 to 0.10] 100.0 [100.0 to 100.0]    77.9 [74.8 to 81.2]
   EGD        Class-weighted argmax   0.62 [0.49 to 0.76]    0.87 [0.80 to 0.91]         0.71 [0.42 to 0.86] 1.00 [1.00 to 1.00]                                      0.15 [0.10 to 0.20] 100.0 [100.0 to 100.0]    79.3 [74.8 to 82.7]
   EGD             Probability sets   0.96 [0.92 to 1.00]    0.98 [0.95 to 1.00]         0.96 [0.92 to 1.00] 1.72 [1.59 to 1.88]                                      0.45 [0.34 to 0.59]    44.3 [33.7 to 53.5]   97.8 [95.2 to 100.0]
   EGD Probability-based abstention   0.99 [0.95 to 1.00]    1.00 [0.99 

Table S8: singleton yield and accuracy of class-conditional conformal prediction at 90%, over the 200 splits (reviewer 2.3).
- Panel A, by true subtype: share of patients given a single subtype, and given the correct single subtype.
- Panel B, by the single subtype given: share of all single-subtype sets, and how often it was correct (precision).

Mean over the 200 test halves [2.5th to 97.5th percentile]. Precision is averaged over the splits in which that subtype was given at least once.

In [10]:
true_rows = []
given_rows = []
for cohort, y in [('EGD', y_egd), ('UTSW', y_utsw)]:
    col = np.searchsorted(classes, y)
    p = np.load(os.path.join(OUT, cohort.lower() + '_probas.npy'))
    is_test = np.load(os.path.join(OUT, 'is_test_' + cohort.lower() + '.npy'))
    scores = 1 - p

    for seed in range(200):
        test = np.where(is_test[seed])[0]
        cal = np.where(~is_test[seed])[0]
        cc = ConformalClassifier()
        cc.fit(scores[cal, col[cal]], bins=y[cal])                     # class-conditional, exactly as in 1_code.ipynb
        sets = np.column_stack([cc.predict_set(scores[test], bins=np.full(len(test), cls), confidence=level, smoothing=False)[:, c]
                                for c, cls in enumerate(classes)])
        yt = col[test]
        single = sets.sum(axis=1) == 1
        given = np.argmax(sets, axis=1)                                 # the subtype named, when the set has one
        correct = single & (given == yt)

        for k in range(3):
            true_rows.append({'cohort': cohort, 'seed': seed, 'subtype': classes[k],
                              'single_pct': single[yt == k].mean() * 100,
                              'correct_single_pct': correct[yt == k].mean() * 100})
            named_k = single & (given == k)
            given_rows.append({'cohort': cohort, 'seed': seed, 'subtype': classes[k],
                               'share_pct': named_k.sum() / single.sum() * 100,
                               'precision_pct': correct[named_k].mean() * 100 if named_k.any() else np.nan})

by_true = pd.DataFrame(true_rows)
by_given = pd.DataFrame(given_rows)

s6a_rows = []
for (cohort, subtype), d in by_true.groupby(['cohort', 'subtype']):
    a = d['single_pct']
    b = d['correct_single_pct']
    s6a_rows.append({'Cohort': cohort, 'True subtype': subtype,
                     'Given a single subtype (%)': '{:.1f} [{:.1f} to {:.1f}]'.format(a.mean(), a.quantile(0.025), a.quantile(0.975)),
                     'Given the correct single subtype (%)': '{:.1f} [{:.1f} to {:.1f}]'.format(b.mean(), b.quantile(0.025), b.quantile(0.975))})

s6b_rows = []
for (cohort, subtype), d in by_given.groupby(['cohort', 'subtype']):
    a = d['share_pct']
    b = d['precision_pct'].dropna()                                     # splits where this subtype was never given are left out
    s6b_rows.append({'Cohort': cohort, 'Subtype given': subtype,
                     'Share of single-subtype sets (%)': '{:.1f} [{:.1f} to {:.1f}]'.format(a.mean(), a.quantile(0.025), a.quantile(0.975)),
                     'Correct when given (%)': '{:.1f} [{:.1f} to {:.1f}]'.format(b.mean(), b.quantile(0.025), b.quantile(0.975)),
                     'Splits where given': len(b)})

tableS8A = pd.DataFrame(s6a_rows)
tableS8B = pd.DataFrame(s6b_rows)
tableS8A.to_csv(os.path.join(OUT, 'tableS8_by_true.csv'), index=False)
tableS8B.to_csv(os.path.join(OUT, 'tableS8_by_given.csv'), index=False)
print(tableS8A.to_string(index=False))
print()
print(tableS8B.to_string(index=False))

Cohort      True subtype Given a single subtype (%) Given the correct single subtype (%)
   EGD       astrocytoma         21.6 [0.0 to 50.0]                   17.1 [0.0 to 44.8]
   EGD      glioblastoma        85.5 [74.0 to 94.5]                  81.0 [71.7 to 89.0]
   EGD oligodendroglioma         21.7 [2.8 to 44.5]                   12.7 [0.0 to 30.6]
  UTSW       astrocytoma          5.6 [0.0 to 28.2]                    2.7 [0.0 to 18.1]
  UTSW      glioblastoma        53.9 [35.9 to 74.9]                  53.2 [35.4 to 72.8]
  UTSW oligodendroglioma         10.0 [0.0 to 32.3]                     2.0 [0.0 to 9.7]

Cohort     Subtype given Share of single-subtype sets (%) Correct when given (%)  Splits where given
   EGD       astrocytoma                8.7 [0.9 to 19.3]    55.4 [0.0 to 100.0]                 198
   EGD      glioblastoma              84.4 [72.4 to 97.3]   98.5 [97.1 to 100.0]                 200
   EGD oligodendroglioma                6.9 [0.0 to 16.6]   58.8 [29.6 to

Table S9: gradient-boosting base learner. Same 20 features, same 200 calibration/test splits and the same class-conditional conformal procedure as 1_code.ipynb.
External AUC on the full cohort. Coverage and mean set size: mean over the 200 test halves, by true subtype, at each target.

In [11]:
targets = [0.80, 0.85, 0.90, 0.95]
s8_auc_rows = []
s8_rows = []
for cohort, y, p in [('EGD', y_egd, egd_gb), ('UTSW', y_utsw, utsw_gb)]:
    s8_auc_rows.append({'Cohort': cohort,
                        'AUC, astrocytoma': round(roc_auc_score(y == classes[0], p[:, 0]), 3),
                        'AUC, glioblastoma': round(roc_auc_score(y == classes[1], p[:, 1]), 3),
                        'AUC, oligodendroglioma': round(roc_auc_score(y == classes[2], p[:, 2]), 3)})
    col = np.searchsorted(classes, y)
    is_test = np.load(os.path.join(OUT, 'is_test_' + cohort.lower() + '.npy'))
    scores = 1 - p

    for seed in range(200):
        test = np.where(is_test[seed])[0]
        cal = np.where(~is_test[seed])[0]
        cc = ConformalClassifier()
        cc.fit(scores[cal, col[cal]], bins=y[cal])
        yt = col[test]
        for target in targets:
            sets = np.column_stack([cc.predict_set(scores[test], bins=np.full(len(test), cls), confidence=target, smoothing=False)[:, c]
                                    for c, cls in enumerate(classes)])
            covered = sets[np.arange(len(test)), yt] == 1
            size = sets.sum(axis=1)
            s8_rows.append({'cohort': cohort, 'seed': seed, 'target': target,
                            'cov_A': covered[yt == 0].mean(), 'cov_G': covered[yt == 1].mean(), 'cov_O': covered[yt == 2].mean(),
                            'size_A': size[yt == 0].mean(), 'size_G': size[yt == 1].mean(), 'size_O': size[yt == 2].mean(),
                            'size': size.mean()})

tableS9_auc = pd.DataFrame(s8_auc_rows)
tableS9 = pd.DataFrame(s8_rows).groupby(['cohort', 'target']).mean(numeric_only=True).drop(columns='seed').round(2).reset_index()
tableS9_auc.to_csv(os.path.join(OUT, 'tableS9_gb_auc.csv'), index=False)
tableS9.to_csv(os.path.join(OUT, 'tableS9_gb_conformal.csv'), index=False)
print(tableS9_auc.to_string(index=False))
print()
print(tableS9.to_string(index=False))

Cohort  AUC, astrocytoma  AUC, glioblastoma  AUC, oligodendroglioma
   EGD             0.919              0.971                   0.865
  UTSW             0.887              0.919                   0.785

cohort  target  cov_A  cov_G  cov_O  size_A  size_G  size_O  size
   EGD    0.80   0.84   0.81   0.80    1.42    0.98    1.33  1.13
   EGD    0.85   0.88   0.85   0.85    1.54    1.08    1.46  1.24
   EGD    0.90   0.93   0.91   0.91    1.67    1.29    1.74  1.44
   EGD    0.95   0.96   0.95   0.97    1.97    1.84    2.14  1.92
  UTSW    0.80   0.82   0.80   0.81    1.59    1.32    1.49  1.38
  UTSW    0.85   0.87   0.85   0.88    1.81    1.58    1.74  1.63
  UTSW    0.90   0.93   0.90   0.91    1.99    1.89    2.05  1.92
  UTSW    0.95   0.98   0.95   0.97    2.24    2.44    2.36  2.40


Table S10: number of features kept by mRMR. Development out-of-fold AUC of the calibrated forest with 10, 20, 30 and 40 features (20 is the main model).

In [12]:
s7_rows = []
for K in k_values:
    p = oof_k[K]
    s7_rows.append({'Features': K,
                    'Macro AUC': round(roc_auc_score(y_dev, p, multi_class='ovr', labels=classes), 3),
                    'AUC, astrocytoma': round(roc_auc_score(y_dev == classes[0], p[:, 0]), 3),
                    'AUC, glioblastoma': round(roc_auc_score(y_dev == classes[1], p[:, 1]), 3),
                    'AUC, oligodendroglioma': round(roc_auc_score(y_dev == classes[2], p[:, 2]), 3)})

tableS10 = pd.DataFrame(s7_rows)
tableS10.to_csv(os.path.join(OUT, 'tableS10_n_features.csv'), index=False)
print(tableS10.to_string(index=False))

 Features  Macro AUC  AUC, astrocytoma  AUC, glioblastoma  AUC, oligodendroglioma
       10      0.925             0.941              0.951                   0.883
       20      0.928             0.949              0.957                   0.878
       30      0.932             0.945              0.956                   0.897
       40      0.934             0.940              0.953                   0.909


Table S11: the 20 features selected by mRMR for the final model, in selection order, with the sequence, image filter and feature class read from each name.

In [13]:
s11_rows = []
for rank, name in enumerate(final_features):
    parts = name.split('_')                      # sequence _ filter _ feature class _ feature
    s11_rows.append({'Rank': rank + 1, 'Sequence': parts[0].replace('T1CE', 'Contrast-enhanced T1'), 'Filter': parts[1],
                     'Feature class': parts[2], 'Feature': parts[3]})
tableS11 = pd.DataFrame(s11_rows)
tableS11.to_csv(os.path.join(OUT, 'tableS11_features.csv'), index=False)
print(tableS11.to_string(index=False))
print()
print(tableS11['Sequence'].value_counts())

 Rank             Sequence              Filter Feature class                          Feature
    1 Contrast-enhanced T1            original    firstorder                     90Percentile
    2                   T1         wavelet-LLL         glszm                      ZoneEntropy
    3                   T2 log-sigma-2-0-mm-3D         glszm GrayLevelNonUniformityNormalized
    4 Contrast-enhanced T1         wavelet-LLH         glszm                      ZoneEntropy
    5 Contrast-enhanced T1            original    firstorder                     10Percentile
    6 Contrast-enhanced T1 log-sigma-1-0-mm-3D          glcm                             Imc2
    7                   T2            original    firstorder                     10Percentile
    8 Contrast-enhanced T1         wavelet-LHL         glszm                      ZoneEntropy
    9 Contrast-enhanced T1 log-sigma-1-0-mm-3D         glszm GrayLevelNonUniformityNormalized
   10                   T1            original    firstorder

In [15]:
import os
os.getcwd()

'g:\\My Drive\\bsd\\11_radiomics_for_gliomas\\02_train\\trial_100\\AJNR submission\\revision1\\code4'

In [1]:
# Table S12 (add as the last cell of 4_supplementary.ipynb): argmax versus class-weighted argmax with development weights, full cohorts.
# Weights chosen on development out-of-fold probabilities (same 729-weight grid as 1_code.ipynb), then applied unchanged.
# Point estimate on the full cohort. 95% CI from 1000 bootstrap resamples of patients, seed 42.
import os
import numpy as np
import pandas as pd
from itertools import product
from sklearn.metrics import balanced_accuracy_score

OUT = 'core_outputs'
table = pd.read_csv('modeling_table.csv', usecols=['site', '2021_glioma_class'])
classes = np.array(['astrocytoma', 'glioblastoma', 'oligodendroglioma'])
y_dev = table[table['site'].isin(['TCGA', 'UCSF'])]['2021_glioma_class'].values
y_egd = table[table['site'] == 'EGD']['2021_glioma_class'].values
y_utsw = table[table['site'] == 'UTSW']['2021_glioma_class'].values
oof = np.load(os.path.join(OUT, 'oof_probas.npy'))
egd = np.load(os.path.join(OUT, 'egd_probas.npy'))
utsw = np.load(os.path.join(OUT, 'utsw_probas.npy'))

# development class weights (same rule as Table S6)
dev_col = np.searchsorted(classes, y_dev)
weight_grid = [np.array(w, dtype=float) for w in product([0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0, 8.0], repeat=3)]
bal = [balanced_accuracy_score(dev_col, np.argmax(oof * w, axis=1)) for w in weight_grid]
dev_w = weight_grid[int(np.argmax(bal))]
print('development class weights (A, G, O):', dev_w)

np.random.seed(42)
rows = []
for cohort, y, p in [('Development (out-of-fold)', y_dev, oof), ('EGD', y_egd, egd), ('UTSW-Glioma', y_utsw, utsw)]:
    col = np.searchsorted(classes, y)
    boot = np.random.randint(0, len(y), (1000, len(y)))
    for rule, pred in [('Argmax', np.argmax(p, axis=1)), ('Class-weighted argmax', np.argmax(p * dev_w, axis=1))]:
        row = {'Cohort': cohort, 'Rule': rule}
        for k in range(3):
            sens = np.mean(pred[col == k] == k)
            spec = np.mean(pred[col != k] != k)
            sens_b = np.full(1000, np.nan)
            spec_b = np.full(1000, np.nan)
            for b in range(1000):
                yb = col[boot[b]]
                pb = pred[boot[b]]
                if np.any(yb == k):
                    sens_b[b] = np.mean(pb[yb == k] == k)
                if np.any(yb != k):
                    spec_b[b] = np.mean(pb[yb != k] != k)
            name = classes[k].capitalize()
            row[name + ' sensitivity'] = '{:.2f} ({:.2f}-{:.2f})'.format(sens, np.nanpercentile(sens_b, 2.5), np.nanpercentile(sens_b, 97.5))
            row[name + ' specificity'] = '{:.2f} ({:.2f}-{:.2f})'.format(spec, np.nanpercentile(spec_b, 2.5), np.nanpercentile(spec_b, 97.5))
        row['Oligodendrogliomas identified'] = str(np.sum((pred == 2) & (col == 2))) + '/' + str(np.sum(col == 2))
        row['Balanced accuracy'] = '{:.2f}'.format(balanced_accuracy_score(col, pred))
        rows.append(row)

tableS12 = pd.DataFrame(rows)
tableS12.to_csv(os.path.join(OUT, 'tableS12_reweighting.csv'), index=False)
print(tableS12.T.to_string())

development class weights (A, G, O): [1.5 0.5 6. ]
                                                       0                          1                 2                      3                 4                      5
Cohort                         Development (out-of-fold)  Development (out-of-fold)               EGD                    EGD       UTSW-Glioma            UTSW-Glioma
Rule                                              Argmax      Class-weighted argmax            Argmax  Class-weighted argmax            Argmax  Class-weighted argmax
Astrocytoma sensitivity                 0.79 (0.72-0.86)           0.72 (0.64-0.80)  0.79 (0.69-0.87)       0.75 (0.65-0.84)  0.70 (0.60-0.81)       0.74 (0.64-0.83)
Astrocytoma specificity                 0.95 (0.93-0.97)           0.96 (0.94-0.97)  0.86 (0.82-0.90)       0.86 (0.82-0.89)  0.92 (0.89-0.94)       0.88 (0.85-0.91)
Glioblastoma sensitivity                0.98 (0.97-0.99)           0.92 (0.90-0.95)  0.96 (0.94-0.98)       0.87 (0.83-